# AutoML-M05: AutoGluon

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Ejecutar **AutoGluon** (Amazon) con multi-layer stacking y ensemble automático
para **Caso D** y **Caso D_strict**.

## 📊 Qué hace AutoGluon

Framework AutoML de Amazon que entrena LightGBM, CatBoost, XGBoost,
Random Forest, Extra Trees, Neural Networks y WeightedEnsemble.
Usa bagging (5-fold) y stacking multi-capa automático.
Con preset `best_quality` prioriza rendimiento sobre velocidad.

## 📝 Nota
Datos auditados (F3-M08). Sin leakage, sin constantes, sin traidoras.

## ⚠️ Requisitos
- **Kernel**: `Python (AutoGluon)` — entorno conda `env_autogluon`
- Creado con `fautoml_setup_entornos.ipynb` (celda 6)

## 📦 Genera
- `data/automl/results_autogluon.parquet` — métricas de todos los modelos
- `data/automl/autogluon_models_D/` — modelos entrenados caso D
- `data/automl/autogluon_models_D_strict/` — modelos entrenados caso D_strict
- `docs/html/fase_automl/m05_autogluon.html` — documentación visual

## 🔄 Flujo
M04 (H2O) → **M05 (AutoGluon)** → M06 (Comparativa)

## 📌 Siguiente
`fautoml_m06_comparativa.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Qué hace: Verifica/instala dependencias, carga librerías, detecta rutas
#   y verifica datasets limpios.
# Requisitos: Kernel env_autogluon (creado con fautoml_setup_entornos.ipynb).
# ============================================================================

import sys, warnings, time, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Auto-verificación de dependencias ---
DEPENDENCIAS_REQUERIDAS = {
    'autogluon': 'autogluon',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'jinja2': 'jinja2',
}

faltantes = []
for modulo, paquete_pip in DEPENDENCIAS_REQUERIDAS.items():
    try:
        __import__(modulo)
    except ImportError:
        faltantes.append(paquete_pip)

if faltantes:
    print(f'⚠️ Instalando dependencias faltantes: {faltantes}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + faltantes
    )
    print(f'✅ Instaladas: {faltantes}')
else:
    print('✅ Todas las dependencias disponibles')

# --- Detección de entorno (Colab / Local) ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

# --- Imports principales ---
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)

# --- Verificar AutoGluon ---
from autogluon.tabular import TabularPredictor
import autogluon
print(f'✅ AutoGluon {autogluon.core.__version__} importado correctamente')

# --- Imports del proyecto ---
from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Rutas y constantes ---
RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
RANDOM_STATE = 42
FRAMEWORK = 'AutoGluon'

# DATASETS: Caso D y D_strict (limpios, sin leakage)
DATASETS = {
    'D': RUTA_AUTOML / 'df_exp_automl_D.parquet',
    'D_strict': RUTA_AUTOML / 'dataset_final_tfm.parquet',
}

info_entorno()

# --- Verificación anti-leakage ---
for caso, ruta in DATASETS.items():
    df_tmp = pd.read_parquet(ruta)
    n_cols = df_tmp.shape[1]
    del df_tmp
    print(f'  ✅ Caso {caso}: {ruta.name} ({n_cols} cols) — LIMPIO')
print('✅ Verificación anti-leakage superada')

⚠️ Instalando dependencias faltantes: ['seaborn']
✅ Instaladas: ['seaborn']
✅ AutoGluon 1.5.0 importado correctamente
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Exce

In [2]:
# ============================================================================
# CELDA 2: EJECUTAR AUTOGLUON (CASO D + D_strict)
# ============================================================================
# Qué hace: Para cada caso, ejecuta TabularPredictor con preset best_quality.
#   AutoGluon hace bagging (5-fold), stacking multi-capa y ensemble.
#   Luego evalúa cada modelo del leaderboard en test set.
# Nota: AutoGluon guarda modelos en disco (carpeta por caso).
# Genera: DataFrame unificado con todas las métricas por caso.
# ============================================================================

TARGET = 'abandono'
all_results = []

for caso, ruta in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'CASO {caso}: {ruta.name}')
    print(f'{"="*70}')

    # --- Cargar datos ---
    df = pd.read_parquet(ruta)
    X = df.drop(columns=[TARGET])
    n_rows = len(df)
    n_cols = len(df.columns)
    n_feat = len(X.columns)
    print(f"Dataset: {n_rows:,} filas × {n_cols} columnas ({n_feat} features)")
    print(f'Target: {(df[TARGET]==1).sum():,} abandono ({(df[TARGET]==1).mean()*100:.1f}%)')

    # --- Split 70/30 estratificado ---
    train_df, test_df = train_test_split(
        df, test_size=0.3, random_state=RANDOM_STATE, stratify=df[TARGET]
    )
    print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

    # --- Ruta para modelos de este caso ---
    # AutoGluon guarda modelos en disco; carpeta separada por caso
    ruta_modelos = RUTA_AUTOML / f'autogluon_models_{caso}'

    # --- Ejecutar AutoGluon ---
    print(f'\n🚀 Ejecutando AutoGluon (best_quality)...')
    print(f'   ⏱️ Tiempo máximo: 10 minutos...')
    t0 = time.time()

    predictor = TabularPredictor(
        label=TARGET,
        path=str(ruta_modelos),
        eval_metric='roc_auc',
        verbosity=2
    ).fit(
        train_data=train_df,
        time_limit=600,
        presets='best_quality',
        auto_stack=True
    )
    t_total = time.time() - t0

    # --- Leaderboard ---
    lb = predictor.leaderboard(test_df, silent=True)
    print(f'\n  ✅ {len(lb)} modelos entrenados en {t_total:.1f}s')
    print(f'\n  Leaderboard (top 5):')
    print(f'  {lb.head(5).to_string(index=False)}')

    # --- Evaluar cada modelo en test set ---
    print(f'\n🔄 Evaluando modelos en test set...')
    y_true = test_df[TARGET].values

    for model_name in lb['model'].values:
        try:
            y_pred = predictor.predict(test_df.drop(columns=[TARGET]), model=model_name)
            y_prob = predictor.predict_proba(test_df.drop(columns=[TARGET]), model=model_name)

            # Probabilidad de la clase positiva (1)
            if 1 in y_prob.columns:
                y_prob_pos = y_prob[1]
            else:
                y_prob_pos = y_prob.iloc[:, 1]

            all_results.append({
                'caso': caso,
                'framework': FRAMEWORK,
                'model_name': model_name,
                'accuracy': accuracy_score(y_true, y_pred),
                'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1': f1_score(y_true, y_pred, zero_division=0),
                'auc_roc': roc_auc_score(y_true, y_prob_pos),
                'mcc': matthews_corrcoef(y_true, y_pred),
                'kappa': cohen_kappa_score(y_true, y_pred),
                'log_loss': round(log_loss(y_true, y_prob_pos), 4),
                'train_time_sec': round(t_total / len(lb), 2),
            })
        except Exception as e:
            print(f'  ⚠️ Error con {model_name}: {e}')

    print(f'  ✅ {len(lb)} modelos evaluados para caso {caso}')

# --- DataFrame unificado ---
df_resultados = pd.DataFrame(all_results)
df_resultados = df_resultados.sort_values(['caso', 'f1'], ascending=[True, False]).reset_index(drop=True)

# --- Ranking final por caso ---
for caso in DATASETS.keys():
    print(f'\n--- RANKING CASO {caso} (por F1, top 10) ---')
    mask = df_resultados['caso'] == caso
    print(df_resultados.loc[mask, ['model_name', 'accuracy', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].head(10).to_string(index=False))

print(f'\n📊 Total: {len(df_resultados)} resultados')

Verbosity: 2 (Standard Logging)



CASO D: df_exp_automl_D.parquet
Dataset: 33,621 filas × 22 columnas (21 features)
Target: 9,833 abandono (29.2%)
Train: 23,534 | Test: 10,087

🚀 Ejecutando AutoGluon (best_quality)...
   ⏱️ Tiempo máximo: 10 minutos...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       8.33 GB / 31.68 GB (26.3%)
Disk Space Avail:   522.74 GB / 951.65 GB (54.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the 


  ✅ 14 modelos entrenados en 618.3s

  Leaderboard (top 5):
                model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
    CatBoost_BAG_L2    0.951177   0.947454     roc_auc       28.002251      30.795313 279.682703                 0.372770                1.350074          14.282347            2       True         13
WeightedEnsemble_L3    0.951071   0.947914     roc_auc       31.057112      32.551541 327.122056                 0.031288                0.007109           1.214354            3       True         14
  LightGBMXT_BAG_L2    0.950925   0.947250     roc_auc       29.315499      29.652062 276.137267                 1.686018                0.206823          10.736911            2       True         10
    LightGBM_BAG_L2    0.950534   0.947052     roc_auc       28.322493      29.612027 291.148508                 0.693012                

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       8.64 GB / 31.68 GB (27.3%)
Disk Space Avail:   521.95 GB / 951.65 GB (54.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluo

  ✅ 14 modelos evaluados para caso D

CASO D_strict: dataset_final_tfm.parquet
Dataset: 33,621 filas × 20 columnas (19 features)
Target: 9,833 abandono (29.2%)
Train: 23,534 | Test: 10,087

🚀 Ejecutando AutoGluon (best_quality)...
   ⏱️ Tiempo máximo: 10 minutos...


Leaderboard on holdout data (DyStack):
                     model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0          LightGBM_BAG_L2       0.935788   0.932068     roc_auc        3.182034       4.226704  68.267202                 0.707093                0.103628           6.133451            2       True          7
1        LightGBMXT_BAG_L2       0.935187   0.932452     roc_auc        2.933629       4.385705  67.237721                 0.458688                0.262629           5.103970            2       True          6
2      WeightedEnsemble_L3       0.934927   0.933503     roc_auc        4.038293       6.032491  77.636096                 0.026063                0.003544           0.510084            3       True          9
3      WeightedEnsemble_L2       0.933534   0.933011     roc_auc        2.502453       4.128141  62.611873               


  ✅ 14 modelos entrenados en 614.2s

  Leaderboard (top 5):
                model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
  LightGBMXT_BAG_L2    0.936682   0.934729     roc_auc       21.530407      10.884917 276.155982                 1.203190                1.000340          23.250664            2       True         10
WeightedEnsemble_L3    0.936662   0.935445     roc_auc       23.866489      14.070603 317.627834                 0.060059                0.001514           1.578404            3       True         14
    CatBoost_BAG_L2    0.936473   0.932550     roc_auc       22.504716      10.563158 261.732213                 2.177498                0.678581           8.826895            2       True         13
    LightGBM_BAG_L2    0.936006   0.934248     roc_auc       21.388740       9.987180 268.988874                 1.061522                

In [3]:
# ============================================================================
# CELDA 3: GUARDAR RESULTADOS
# ============================================================================
# Qué hace: Guarda métricas en parquet para la comparativa final (M06).
# Genera: data/automl/results_autogluon.parquet
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_autogluon.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} filas (caso D + D_strict)')

💾 results_autogluon.parquet: 28 filas (caso D + D_strict)


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS Y HTML
# ============================================================================
# Qué hace: Genera gráficos comparativos y documentación HTML.
# Genera: docs/html/fase_automl/m05_autogluon.html
# ============================================================================

graficos_b64 = {}
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='m05'
)

# --- Gráfico 1: Top 10 modelos por F1 (barras horizontales) ---
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(10).copy()
    df_plot = df_plot.sort_values('f1', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = np.arange(len(df_plot))

    bars = ax.barh(y_pos, df_plot['f1'], color='#ed8936', alpha=0.85, height=0.6)
    ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=50, zorder=5, label='AUC-ROC')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['model_name'], fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'AutoGluon Caso {caso}: Top 10 Modelos', fontweight='bold', fontsize=14)
    ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)

    for bar, val in zip(bars, df_plot['f1']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8, color='#2d3748')

    plt.tight_layout()
    graficos_b64[f'top10_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Gráfico 2: Top 5 métricas detalladas ---
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_plot = df_resultados[mask].head(5).copy()

    metricas_plot = ['accuracy', 'f1', 'auc_roc', 'mcc', 'kappa']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_plot))
    width = 0.15
    colores = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

    for j, (met, col) in enumerate(zip(metricas_plot, colores)):
        ax.bar(x + j*width, df_plot[met], width, label=met, color=col)

    ax.set_xticks(x + width*2)
    ax.set_xticklabels(df_plot['model_name'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'AutoGluon Caso {caso}: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    graficos_b64[f'barras_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Construir HTML ---
contenido_html = ''

contenido_html += generar_seccion_html('Metodología', f'''
<p><strong>AutoGluon</strong> (Amazon) es un framework AutoML que entrena LightGBM,
CatBoost, XGBoost, Random Forest, Extra Trees, Neural Networks y
WeightedEnsemble con bagging y stacking multi-capa automáticos.</p>
<div style="background:#fffaf0;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #ed8936;">
  <strong>⚙️ Config:</strong> preset=best_quality, time_limit=600s, auto_stack=True,
  eval_metric=roc_auc, verbosity=2.
</div>
<div style="background:#EBF8FF;padding:12px;border-radius:8px;margin-top:10px;border-left:4px solid #3182ce;">
  <strong>📌 Evaluación:</strong> Métricas calculadas sobre test set (30%) con predicciones
  y probabilidades de cada modelo individual.
</div>
''', '🔬')

# Resultados por caso
for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_c = df_resultados[mask]
    mejor = df_c.iloc[0]
    n_modelos = len(df_c)

    # Tabla
    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    tabla += '<tr style="background:#ed8936;color:white;">'
    for col in ['#', 'Modelo', 'Acc', 'Prec', 'Recall', 'F1', 'AUC', 'MCC', 'Kappa', 'Tiempo']:
        tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
    tabla += '</tr>\n'

    for rank, (i, row) in enumerate(df_c.iterrows(), 1):
        bg = '#f7fafc' if rank % 2 == 0 else 'white'
        if rank <= 3:
            medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
            rank_str = medallas[rank]
        else:
            rank_str = str(rank)

        tabla += f'<tr style="background:{bg};">'
        tabla += f'<td style="padding:6px;text-align:center;font-size:11px;">{rank_str}</td>'
        tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
        for c in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'mcc', 'kappa']:
            v = row[c]
            color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["train_time_sec"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    graf_top10 = f'<img src="data:image/png;base64,{graficos_b64[f"top10_{caso}"]}" style="max-width:100%;border-radius:8px;">'
    graf_barras = f'<img src="data:image/png;base64,{graficos_b64[f"barras_{caso}"]}" style="max-width:100%;border-radius:8px;">'

    desc_caso = 'Alerta temprana (21 features)' if caso == 'D' else 'Producción (19 features)'
    contenido_html += generar_seccion_html(
        f'🚀 Caso {caso}: {desc_caso}',
        f'<p><strong>🏆 Mejor:</strong> {mejor["model_name"]} '
        f'(F1={mejor["f1"]:.4f}, AUC={mejor["auc_roc"]:.4f}, MCC={mejor["mcc"]:.4f})</p>'
        f'<p><strong>📊 Modelos evaluados:</strong> {n_modelos}</p>\n'
        f'{graf_top10}\n<br>\n{tabla}\n<br>\n{graf_barras}'
    )

# --- Comparativa acumulada M01-M05 ---
frameworks_prev = []
for fname, flabel in [('results_baselines.parquet', 'Baselines (M01)'),
                       ('results_lazypredict.parquet', 'LazyPredict (M02)'),
                       ('results_pycaret.parquet', 'PyCaret (M03)'),
                       ('results_h2o.parquet', 'H2O (M04)')]:
    ruta_fw = RUTA_AUTOML / fname
    if ruta_fw.exists():
        frameworks_prev.append((flabel, pd.read_parquet(ruta_fw)))

if frameworks_prev:
    comparativa = '<table style="width:100%;border-collapse:collapse;">\n'
    comparativa += '<tr style="background:#ed8936;color:white;">'
    for col in ['Caso', 'Framework', 'Mejor Modelo', 'F1', 'AUC', 'MCC']:
        comparativa += f'<th style="padding:8px;text-align:center;">{col}</th>'
    comparativa += '</tr>\n'

    for caso in DATASETS.keys():
        for fw_name, df_fw in frameworks_prev:
            mask_fw = df_fw['caso'] == caso
            if 'model_name' in df_fw.columns:
                mask_fw = mask_fw & (~df_fw['model_name'].str.startswith('Dummy'))
            df_fw_caso = df_fw[mask_fw].sort_values('f1', ascending=False)
            if len(df_fw_caso) > 0:
                mejor_fw = df_fw_caso.iloc[0]
                comparativa += f'<tr style="background:#f7fafc;">'
                comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
                comparativa += f'<td style="padding:6px;text-align:center;">{fw_name}</td>'
                comparativa += f'<td style="padding:6px;">{mejor_fw["model_name"]}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
                comparativa += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td></tr>\n'

        # AutoGluon (actual)
        mask_ag = df_resultados['caso'] == caso
        mejor_ag = df_resultados[mask_ag].iloc[0]
        comparativa += f'<tr style="background:white;">'
        comparativa += f'<td style="padding:6px;text-align:center;">{caso}</td>'
        comparativa += f'<td style="padding:6px;text-align:center;"><strong>AutoGluon (M05)</strong></td>'
        comparativa += f'<td style="padding:6px;"><strong>{mejor_ag["model_name"]}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_ag["f1"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_ag["auc_roc"]:.4f}</strong></td>'
        comparativa += f'<td style="text-align:center;"><strong>{mejor_ag["mcc"]:.4f}</strong></td></tr>\n'

    comparativa += '</table>'
    contenido_html += generar_seccion_html('🔄 Comparativa: M01 vs M02 vs M03 vs M04 vs M05', comparativa)

# --- KPIs ---
casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_d = len(df_resultados[df_resultados['caso']==casos_k[0]])

KPIS = [
    {'valor': str(n_modelos_d), 'titulo': 'Modelos'},
    {'valor': f'{mejor_d["f1"]:.4f}', 'titulo': f'Mejor F1 (D)'},
    {'valor': f'{mejor_ds["f1"]:.4f}', 'titulo': f'Mejor F1 (D_strict)'},
    {'valor': 'Stacking', 'titulo': 'Técnica'},
]

# --- Renderizar HTML ---
html = render_base_html(
    titulo='M05: AutoGluon', icono='🚀',
    subtitulo=f'Multi-layer Stacking (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido_html,
    notebook_nombre='fautoml_m05_autogluon.ipynb', notebook_carpeta='fase_automl'
)
ruta_html = RUTA_FASE_AUTOML_HTML / 'm05_autogluon.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m05_autogluon.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m05_autogluon.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN
# ============================================================================

casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
n_modelos_d = len(df_resultados[df_resultados['caso']==casos_k[0]])

print('\n' + '='*60)
print('✅ AUTOML-M05 COMPLETADO')
print('='*60)
print(f'Framework: AutoGluon {autogluon.core.__version__}')
print(f'Técnica: Multi-layer stacking + bagging')
print(f'Modelos caso D: {n_modelos_d}')
print(f'Caso D mejor: {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f}, AUC={mejor_d["auc_roc"]:.4f})')
print(f'Caso D_strict mejor: {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f}, AUC={mejor_ds["auc_roc"]:.4f})')
print(f'Resultados: {ruta_resultados}')
print(f'HTML: {ruta_html}')
print(f'\n🎯 Siguiente: fautoml_m06_comparativa.ipynb')
print('='*60)


✅ AUTOML-M05 COMPLETADO
Framework: AutoGluon 1.5.0
Técnica: Multi-layer stacking + bagging
Modelos caso D: 14
Caso D mejor: CatBoost_BAG_L2 (F1=0.8138, AUC=0.9512)
Caso D_strict mejor: CatBoost_BAG_L2 (F1=0.7970, AUC=0.9365)
Resultados: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_autogluon.parquet
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m05_autogluon.html

🎯 Siguiente: fautoml_m06_comparativa.ipynb
